# Azure Container Registry (ACR) & Azure Container Instances (ACI) Deployment\nThis notebook demonstrates how to deploy a Dockerized ML application to Azure.\n

In [ ]:
!pip install azure-storage-blob azure-identity azure-mgmt-containerinstance\n

## 1. Upload Model Artifact to Azure Blob Storage\nEnsure your model artifacts are stored in Azure Blob Storage so the container can download them at runtime.\n

In [ ]:
from azure.identity import DefaultAzureCredential\nfrom azure.storage.blob import BlobServiceClient\n\naccount_url = 'https://<your-storage-account>.blob.core.windows.net'\ncredential = DefaultAzureCredential()\nblob_service_client = BlobServiceClient(account_url=account_url, credential=credential)\ncontainer_name = 'mlops-data'\n\ndef upload_model():\n    blob_client = blob_service_client.get_blob_client(container=container_name, blob='models/model.pt')\n    with open('model.pt', 'rb') as data:\n        blob_client.upload_blob(data, overwrite=True)\n    print('Model uploaded to Azure Blob Storage.')\n\n# upload_model()\n

## 2. Push to Azure Container Registry (ACR)\n

In [ ]:
# Build and push the image using az cli and docker\n!az acr login --name YourRegistryName\n!docker build -t mlops-api .\n!docker tag mlops-api yourregistryname.azurecr.io/mlops-api:latest\n!docker push yourregistryname.azurecr.io/mlops-api:latest\n

## 3. Deploy to Azure Container Instances (ACI)\n

In [ ]:
!az container create \\\n    --resource-group mlops-rg \\\n    --name mlops-api-container \\\n    --image yourregistryname.azurecr.io/mlops-api:latest \\\n    --dns-name-label mlops-api-instance \\\n    --ports 80 \\\n    --cpu 1 \\\n    --memory 1.5\n